# 05 — Catalog Enrichment (Tags, Comments, Lineage)

This notebook is the **Databricks One prep** — it adds the governance metadata that
makes tables discoverable and meaningful in Discover, Domains, and Databricks One.

**What we do:**
1. `COMMENT ON` — descriptions on catalogs, schemas, tables, and columns
2. `SET TAGS` — domain, tier, PII, data owner tags on tables and columns
3. Verify via `information_schema.table_tags` and `information_schema.column_tags`
4. Query `system.access.table_lineage` to confirm lineage was captured from notebooks 02–04

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "nil_sponsorships")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

print(f"Catalog: {UC_CATALOG}")
print(f"Schemas: {BRONZE_SCHEMA}, {SILVER_SCHEMA}, {GOLD_SCHEMA}")

Catalog: alexander_booth
Schemas: nil_sponsorships_bronze, nil_sponsorships_silver, nil_sponsorships_gold


## 1. Schema-Level Comments

These appear in Catalog Explorer and Discover when users browse schemas.

In [2]:
comments = {
    f"{UC_CATALOG}.{BRONZE_SCHEMA}": "Raw ingested NIL sponsorship data. VARIANT-based Delta tables loaded via Auto Loader from UC Volumes.",
    f"{UC_CATALOG}.{SILVER_SCHEMA}": "Cleaned and typed NIL sponsorship data. Deduplicated with MD5 surrogate keys and PK/FK constraints.",
    f"{UC_CATALOG}.{GOLD_SCHEMA}": "Analytics-ready star schema for NIL sponsorship insights. Dimensions, facts, and pre-aggregated views for dashboards and Genie.",
}

for schema, comment in comments.items():
    spark.sql(f"COMMENT ON SCHEMA {schema} IS '{comment}'")
    print(f"Commented: {schema}")

print("\nSchema comments applied.")

Commented: alexander_booth.nil_sponsorships_bronze
Commented: alexander_booth.nil_sponsorships_silver
Commented: alexander_booth.nil_sponsorships_gold

Schema comments applied.


## 2. Table-Level Comments

These are surfaced in Discover, Genie, and AI/BI dashboards for context.

In [3]:
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"

table_comments = {
    f"{G}.dim_athlete": "Dimension table for college athletes participating in NIL deals. Contains school, sport, position, and social media following metrics.",
    f"{G}.dim_sponsor": "Dimension table for brands sponsoring college athletes. Includes industry, budget tier, and geographic region.",
    f"{G}.dim_date": "Standard date dimension (2024-2027) with fiscal and calendar attributes for time-series analysis.",
    f"{G}.fact_deals": "Fact table at deal grain. Each row is one NIL contract between an athlete and sponsor, with deal amount, type, status, and duration.",
    f"{G}.fact_campaign_performance": "Fact table at campaign-event grain. Tracks impressions, engagements, clicks, conversions, and spend for each marketing campaign tied to a deal.",
}

for table, comment in table_comments.items():
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")
    print(f"Commented: {table}")

print("\nTable comments applied.")

Commented: alexander_booth.nil_sponsorships_gold.dim_athlete
Commented: alexander_booth.nil_sponsorships_gold.dim_sponsor
Commented: alexander_booth.nil_sponsorships_gold.dim_date
Commented: alexander_booth.nil_sponsorships_gold.fact_deals
Commented: alexander_booth.nil_sponsorships_gold.fact_campaign_performance

Table comments applied.


## 3. Column-Level Comments

Key columns get descriptions so Genie and business users understand the data.

In [4]:
column_comments = [
    # dim_athlete
    (f"{G}.dim_athlete", "athlete_sk", "MD5 surrogate key derived from athlete_id"),
    (f"{G}.dim_athlete", "total_followers", "Sum of Instagram, TikTok, and Twitter followers"),
    (f"{G}.dim_athlete", "conference", "Athletic conference (SEC, Big Ten, Big 12, ACC, Pac-12)"),
    (f"{G}.dim_athlete", "year", "Academic year: Freshman, Sophomore, Junior, or Senior"),
    # dim_sponsor
    (f"{G}.dim_sponsor", "budget_tier", "Sponsor budget category: Starter, Growth, or Enterprise"),
    (f"{G}.dim_sponsor", "industry", "Sponsor industry vertical: Apparel, Food & Beverage, Technology, Automotive, or Finance"),
    # fact_deals
    (f"{G}.fact_deals", "deal_amount", "Total contract value in USD"),
    (f"{G}.fact_deals", "deal_type", "NIL deal category: Social Media Post, Autograph Signing, Brand Ambassador, Merchandise Licensing, Event Appearance, or Content Creation"),
    (f"{G}.fact_deals", "status", "Deal lifecycle status: Pending, Active, Completed, or Cancelled"),
    (f"{G}.fact_deals", "deal_duration_days", "Number of days between deal start and end dates"),
    # fact_campaign_performance
    (f"{G}.fact_campaign_performance", "impressions", "Total number of times the campaign content was displayed"),
    (f"{G}.fact_campaign_performance", "engagements", "Total interactions (likes, comments, shares) with campaign content"),
    (f"{G}.fact_campaign_performance", "conversions", "Number of completed desired actions (purchases, sign-ups) attributed to the campaign"),
    (f"{G}.fact_campaign_performance", "cost_per_click", "Campaign spend divided by total clicks (USD)"),
    (f"{G}.fact_campaign_performance", "platform", "Marketing platform: Instagram, TikTok, Twitter, YouTube, or In-Person"),
]

for table, column, comment in column_comments:
    escaped = comment.replace("'", "\\'")
    spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} COMMENT '{escaped}'")
    print(f"  {table.split('.')[-1]}.{column}")

print(f"\n{len(column_comments)} column comments applied.")

  dim_athlete.athlete_sk
  dim_athlete.total_followers
  dim_athlete.conference
  dim_athlete.year
  dim_sponsor.budget_tier
  dim_sponsor.industry
  fact_deals.deal_amount
  fact_deals.deal_type
  fact_deals.status
  fact_deals.deal_duration_days
  fact_campaign_performance.impressions
  fact_campaign_performance.engagements
  fact_campaign_performance.conversions
  fact_campaign_performance.cost_per_click
  fact_campaign_performance.platform

15 column comments applied.


## 4. Tags — Domain, Tier, PII, Owner

Domains are backed by a governed tag **key**. When you create a governed tag
`nil_sponsorships` and build a domain from it, every asset carrying the tag
key `nil_sponsorships` automatically appears in that domain.

We apply the domain tag key here as an ungoverned tag. When you create the
governed tag in the UI (notebook 08, Step 2), it converts these existing
assignments to governed and the assets auto-populate in the domain.

In [5]:
# Tag all gold tables with the domain tag key + metadata tags
gold_tables = [
    "dim_athlete", "dim_sponsor", "dim_date",
    "fact_deals", "fact_campaign_performance",
]

for table in gold_tables:
    full = f"{UC_CATALOG}.{GOLD_SCHEMA}.{table}"
    spark.sql(f"""
        ALTER TABLE {full} SET TAGS (
            'nil_sponsorships' = '',
            'tier' = 'gold',
            'data_owner' = 'nil_analytics_team'
        )
    """)
    print(f"  Tagged: {table}")

print("\nGold table tags applied.")

  Tagged: dim_athlete
  Tagged: dim_sponsor
  Tagged: dim_date
  Tagged: fact_deals
  Tagged: fact_campaign_performance

Gold table tags applied.


In [6]:
# Tag silver tables with the domain tag key + tier
silver_tables = ["athletes", "sponsors", "deals", "campaigns"]

for table in silver_tables:
    full = f"{UC_CATALOG}.{SILVER_SCHEMA}.{table}"
    spark.sql(f"""
        ALTER TABLE {full} SET TAGS (
            'nil_sponsorships' = '',
            'tier' = 'silver',
            'data_owner' = 'nil_analytics_team'
        )
    """)
    print(f"  Tagged: {table}")

print("\nSilver table tags applied.")

  Tagged: athletes
  Tagged: sponsors
  Tagged: deals
  Tagged: campaigns

Silver table tags applied.


In [7]:
# Tag PII columns on silver.athletes and financial columns on fact_deals
# Note: tag keys may be governed in your workspace with restricted allowed values.
# We wrap each tag in a try/except so the notebook keeps running if a tag conflicts.

column_tags = [
    (f"{UC_CATALOG}.{SILVER_SCHEMA}.athletes", "email", "contains_pii", "y"),
    (f"{UC_CATALOG}.{SILVER_SCHEMA}.athletes", "phone", "contains_pii", "y"),
    (f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_deals", "deal_amount", "data_sensitivity", "financial"),
]

for table, col, tag_key, tag_val in column_tags:
    try:
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN {col} SET TAGS ('{tag_key}' = '{tag_val}')")
        print(f"  Tagged: {table.split('.')[-1]}.{col} ({tag_key}={tag_val})")
    except Exception as e:
        err = str(e)
        if "INVALID_PARAMETER_VALUE" in err or "not an allowed value" in err:
            print(f"  Skipped: {table.split('.')[-1]}.{col} — '{tag_key}' is a governed tag with restricted values")
            print(f"           Adjust the value to match your workspace's allowed values.")
        else:
            raise

print("\nColumn-level tags applied.")

  Tagged: athletes.email (contains_pii=y)
  Tagged: athletes.phone (contains_pii=y)
  Skipped: fact_deals.deal_amount — 'data_sensitivity' is a governed tag with restricted values
           Adjust the value to match your workspace's allowed values.

Column-level tags applied.


## 5. Verify Tags via information_schema

In [8]:
print("Table tags:")
spark.sql(f"""
    SELECT schema_name, table_name, tag_name, tag_value
    FROM {UC_CATALOG}.information_schema.table_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
    ORDER BY schema_name, table_name, tag_name
""").show(100, truncate=False)

print("\nColumn tags (PII + financial):")
spark.sql(f"""
    SELECT schema_name, table_name, column_name, tag_name, tag_value
    FROM {UC_CATALOG}.information_schema.column_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
    ORDER BY schema_name, table_name, column_name
""").show(100, truncate=False)

Table tags:
+-----------------------+-------------------------+----------------+------------------+
|schema_name            |table_name               |tag_name        |tag_value         |
+-----------------------+-------------------------+----------------+------------------+
|nil_sponsorships_gold  |dim_athlete              |data_owner      |nil_analytics_team|
|nil_sponsorships_gold  |dim_athlete              |nil_sponsorships|                  |
|nil_sponsorships_gold  |dim_athlete              |tier            |gold              |
|nil_sponsorships_gold  |dim_date                 |data_owner      |nil_analytics_team|
|nil_sponsorships_gold  |dim_date                 |nil_sponsorships|                  |
|nil_sponsorships_gold  |dim_date                 |tier            |gold              |
|nil_sponsorships_gold  |dim_sponsor              |data_owner      |nil_analytics_team|
|nil_sponsorships_gold  |dim_sponsor              |nil_sponsorships|                  |
|nil_sponsorships_go

## 6. Verify Lineage

The lineage system table captures runtime lineage from notebooks 02–04.
This shows the Bronze → Silver → Gold data flow automatically.

In [9]:
print("Table lineage (targets in this demo):")
print("Note: lineage may take several minutes to propagate after running notebooks 02-04.")
spark.sql(f"""
    SELECT source_table_full_name, target_table_full_name, entity_type, created_by,
           event_time
    FROM system.access.table_lineage
    WHERE target_table_full_name LIKE '{UC_CATALOG}.{UC_SCHEMA}_%'
    ORDER BY event_time DESC
    LIMIT 20
""").show(truncate=False)

Table lineage (targets in this demo):
Note: lineage may take several minutes to propagate after running notebooks 02-04.
+---------------------------------------------------------------+----------------------------------------------------------------+-----------+------------------------------+-----------------------+
|source_table_full_name                                         |target_table_full_name                                          |entity_type|created_by                    |event_time             |
+---------------------------------------------------------------+----------------------------------------------------------------+-----------+------------------------------+-----------------------+
|alexander_booth.nil_sponsorships_gold.dim_sponsor              |alexander_booth.nil_sponsorships_gold.v_sponsor_roi             |NULL       |alexander.booth@databricks.com|2026-04-07 18:45:53.828|
|alexander_booth.nil_sponsorships_gold.fact_campaign_performance|alexander_booth.nil_sp

In [10]:
print("=" * 60)
print("CATALOG ENRICHMENT SUMMARY")
print("=" * 60)

tag_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.information_schema.table_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
""").collect()[0]["cnt"]

col_tag_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.information_schema.column_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
""").collect()[0]["cnt"]

print(f"  Table tags applied:  {tag_count}")
print(f"  Column tags applied: {col_tag_count}")
print(f"  Schema comments:     3")
print(f"  Table comments:      {len(table_comments)}")
print(f"  Column comments:     {len(column_comments)}")
print("\nCatalog enrichment complete.")
print("Next: Create dashboard (06) and Genie space (07), then follow the Discover walkthrough (08).")

CATALOG ENRICHMENT SUMMARY
  Table tags applied:  27
  Column tags applied: 2
  Schema comments:     3
  Table comments:      5
  Column comments:     15

Catalog enrichment complete.
Next: Create dashboard (06) and Genie space (07), then follow the Discover walkthrough (08).
